In [1]:
import pandas as frame


In [2]:
orig = frame.read_csv('/Users/macbookpro/platform/Backend/data/working/origination_v02.csv')

/var/folders/gy/2mbpv0kx5jz22fry28cqlnkc0000gn/T/ipykernel_64626/3542022219.py:1: DtypeWarning: Columns (25,26,30) have mixed types. Specify dtype option on import or set low_memory=False.
  orig = frame.read_csv('/Users/macbookpro/platform/Backend/data/working/origination_v02.csv')


In [3]:
hist  = frame.read_csv('/Users/macbookpro/platform/Backend/data/working/hist_12m.csv')

/var/folders/gy/2mbpv0kx5jz22fry28cqlnkc0000gn/T/ipykernel_64626/2861163649.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  hist  = frame.read_csv('/Users/macbookpro/platform/Backend/data/working/hist_12m.csv')


In [10]:
import importlib
import pipelines.Features.window_builder as window_builder
importlib.reload(window_builder)


import pipelines.Features.featurePipeline as pipeline
importlib.reload(pipeline)

from pipelines.Features.featurePipeline import FeaturePipeline

from pipelines.Features.window_builder import WindowBuilder
from pipelines.Features.delinquency_features import DelinquencyFeatures
from pipelines.Features.capital_features import CapitalFeatures
from pipelines.Features.origination_features import OriginationFeatures

In [ ]:
import time

t0 = time.time()
hist_12m = WindowBuilder(hist, window_months=12).build()
print(f"WindowBuilder      : {time.time()-t0:.1f}s")

t0 = time.time()
delinquency_agg = DelinquencyFeatures(hist_12m).build()
print(f"DelinquencyFeatures: {time.time()-t0:.1f}s")

t0 = time.time()
capital_agg = CapitalFeatures(hist_12m, orig_df=orig).build()
print(f"CapitalFeatures    : {time.time()-t0:.1f}s")

t0 = time.time()
orig_agg = OriginationFeatures(orig).build()
print(f"OriginationFeatures: {time.time()-t0:.1f}s")

In [45]:
import dask.dataframe as dd
import time

In [12]:
t0 = time.time()
data = dd.from_pandas(hist, npartitions=8)
print(f"Dask DataFrame     : {time.time()-t0:.1f}s")


Dask DataFrame     : 42.0s


In [13]:
t0 = time.time()
essai = hist.copy()
print(f"Pandas Copy        : {time.time()-t0:.1f}s")


Pandas Copy        : 4.8s


In [14]:
hist.head()

,LOAN_SEQUENCE_NUMBER,MONTHLY_REPORTING_PERIOD,CURRENT_ACTUAL_UPB,CURRENT_LOAN_DELINQUENCY_STATUS,LOAN_AGE,REMAINING_MONTHS_TO_LEGAL_MATURITY,MODIFICATION_FLAG,ZERO_BALANCE_CODE,ZERO_BALANCE_EFFECTIVE_DATE,CURRENT_INTEREST_RATE,CURRENT_NON_INTEREST_BEARING_UPB,DUE_DATE_OF_LAST_PAID_INSTALLMENT,INTEREST_RATE_STEP_INDICATOR,ESTIMATED_LTV,DELINQUENCY_DUE_TO_DISASTER,BORROWER_ASSISTANCE_STATUS_CODE,INTEREST_BEARING_UPB,VINTAGE,DPD_DAYS
0,F07Q10000001,2007-05-01,168000.0,0,1,239.0,N,0.0,NaN,7.375,0.0,NaN,N,74.556213,N,N,168000.0,2007Q1,0.0
1,F07Q10000001,2007-06-01,168000.0,0,2,238.0,N,0.0,NaN,7.375,0.0,NaN,N,74.556213,N,N,168000.0,2007Q1,0.0
2,F07Q10000001,2007-07-01,168000.0,0,3,237.0,N,0.0,NaN,7.375,0.0,NaN,N,74.556213,N,N,168000.0,2007Q1,0.0
3,F07Q10000001,2007-08-01,167000.0,0,4,236.0,N,0.0,NaN,7.375,0.0,NaN,N,74.112426,N,N,167000.0,2007Q1,0.0
4,F07Q10000001,2007-09-01,167000.0,0,5,235.0,N,0.0,NaN,7.375,0.0,NaN,N,74.112426,N,N,167000.0,2007Q1,0.0


In [46]:
t0 = time.time()
delinquency_agg = DelinquencyFeatures(hist).build()
print(f"DelinquencyFeatures: {time.time()-t0:.1f}s")

Copy DataFrame     : 5.2s
Cast DPD           : 6.4s
Groupby            : 0.0s
Colonnes de travail: 3.5s
DelinquencyFeatures: 27.1s


In [47]:
delinquency_agg["tendance"].describe()


count    1.390113e+06
mean    -2.959640e-03
std      2.253981e-01
min     -1.938811e+01
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      4.650350e+00
Name: tendance, dtype: float64

In [48]:
delinquency_agg["tendance"].isna().sum()

np.int64(857)

In [45]:
delinquency_agg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1390970 entries, 0 to 1390969
Data columns (total 11 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   LOAN_SEQUENCE_NUMBER   1390970 non-null  object 
 1   freq                   1390970 non-null  float64
 2   severite               1390970 non-null  float64
 3   profondeur_max         1390970 non-null  float64
 4   n_profondeur_max       1390970 non-null  int64  
 5   tendance               1390113 non-null  float64
 6   recuperation           89052 non-null    float64
 7   freq_x_profondeur_max  1390970 non-null  float64
 8   freq_x_tendance        1390113 non-null  float64
 9   freq_x_recuperation    89052 non-null    float64
 10  recidivisme_extreme    1390970 non-null  float64
dtypes: float64(9), int64(1), object(1)
memory usage: 116.7+ MB


In [63]:
t0 = time.time()
capital_agg = CapitalFeatures(hist, orig_df=orig).build()
print(f"CapitalFeatures    : {time.time()-t0:.1f}s")

CapitalFeatures    : 35.5s


In [64]:
capital_agg.head()

,LOAN_SEQUENCE_NUMBER,niveau,progression,ecart_au_plan,anticipation
0,F07Q10000001,0.000000,49621.425749,165152.653246,0.083333
1,F07Q10000002,0.000000,3.771474,0.000000,0.000000
2,F07Q10000003,0.861902,5.698937,-16660.654917,0.000000
3,F07Q10000004,0.000000,175.203789,8413.803387,0.083333
4,F07Q10000005,0.000000,33592.307319,113074.960468,0.083333


In [65]:
t0 = time.time()
orig_agg = OriginationFeatures(orig).build()
print(f"OriginationFeatures: {time.time()-t0:.1f}s")

OriginationFeatures: 2.9s


In [69]:
hist.shape

(15041114, 19)

In [68]:
window_builder = window_builder.WindowBuilder(hist).build()

In [70]:
window_builder.shape

(15041114, 19)

1.8.0
0.20.0


In [7]:
pipeline   = FeaturePipeline(window_months=12)
X, y       = pipeline.fit_transform(hist, orig)

Copy DataFrame     : 8.0s
Cast DPD           : 6.2s
Groupby            : 0.0s
Colonnes de travail: 6.9s


In [9]:
X.head()

,mensualite_implicite,charge_taux_duree,pression_levier,ecart_ltv_ocltv,refi_flag,credit_segment,co_emprunteur,freq,niveau,ecart_au_plan,anticipation
0,0.050111,0.363241,-0.059947,0.101435,-0.399800,0.469573,0.299530,3.583997,-0.350316,-0.329721,-0.540267
1,-0.600446,0.992396,0.172662,0.101435,-0.399800,-1.082225,-0.243918,3.583997,-0.350316,3.279838,2.367063
2,0.065494,0.277890,-0.872189,0.101435,0.181599,-1.082225,-0.243918,3.583997,1.598231,0.232127,2.367063
3,-0.225926,0.992396,0.571477,0.101435,-0.399800,-1.082225,-0.243918,3.583997,-0.350316,1.280242,-0.540267
4,-0.315223,-0.842447,-0.872189,0.101435,0.181599,0.469573,-0.243918,3.583997,-0.350316,-0.329721,-0.540267


In [10]:
# 1. Shape
print(X.shape, y.shape)

# 2. Pas de NaN
print(X.isna().sum().sum())
print(y.isna().sum())

# 3. Distribution de la target
print(y.value_counts(normalize=True))

# 4. Aperçu des features
print(X.describe())

# 5. Cohérence index
print(X.index.equals(y.index))

(1390970, 11) (1390970,)
0
0
default
0    0.966682
1    0.033318
Name: proportion, dtype: float64
       mensualite_implicite  charge_taux_duree  pression_levier  \
count          1.390970e+06       1.390970e+06     1.390970e+06   
mean           1.064838e-01       2.963928e-01     1.325916e-01   
std            5.203072e-01       7.941377e-01     5.524776e-01   
min           -6.004455e-01      -1.103018e+00    -8.721885e-01   
25%           -2.259258e-01      -5.248888e-01    -3.136227e-01   
50%           -5.144074e-02       3.632410e-01     7.640314e-02   
75%            2.395604e-01       8.590634e-01     5.714770e-01   
max            1.321448e+00       1.645899e+00     1.260742e+00   

       ecart_ltv_ocltv     refi_flag  credit_segment  co_emprunteur  \
count     1.390970e+06  1.390970e+06    1.390970e+06   1.390970e+06   
mean      3.549325e-02  3.237125e-02    2.033137e-01   3.422590e-02   
std       2.449534e-01  2.539527e-01    5.850527e-01   2.716487e-01   
min      -8.74

In [11]:
pipeline.woe_pipeline_.selection_report()

{'iv_threshold': 0.02,
 'selected': ['mensualite_implicite',
  'charge_taux_duree',
  'pression_levier',
  'ecart_ltv_ocltv',
  'refi_flag',
  'credit_segment',
  'co_emprunteur',
  'freq',
  'niveau',
  'ecart_au_plan',
  'anticipation'],
 'rejected': ['couverture_mi',
  'multi_unite',
  'occupancy_risk',
  'primo_accedant',
  'tendance',
  'recuperation',
  'profondeur_max',
  'n_profondeur_max',
  'freq_x_profondeur_max']}

In [84]:
pipeline   = FeaturePipeline(window_months=12,woe_config= {"iv_threshold": 0.01, "metric": "woe"})
X, y       = pipeline.fit_transform(hist, orig)

Copy DataFrame     : 5.7s
Cast DPD           : 6.4s
Groupby            : 0.0s
Colonnes de travail: 8.1s


In [13]:
pipeline.woe_pipeline_.selection_report()

{'iv_threshold': 0.01,
 'selected': ['mensualite_implicite',
  'charge_taux_duree',
  'pression_levier',
  'ecart_ltv_ocltv',
  'couverture_mi',
  'refi_flag',
  'credit_segment',
  'primo_accedant',
  'co_emprunteur',
  'freq',
  'niveau',
  'ecart_au_plan',
  'anticipation'],
 'rejected': ['multi_unite',
  'occupancy_risk',
  'tendance',
  'recuperation',
  'profondeur_max',
  'n_profondeur_max',
  'freq_x_profondeur_max']}

In [43]:
X.tendance.unique()

array([None], dtype=object)

In [85]:
X = pipeline.transform(hist, orig)

Copy DataFrame     : 7.0s
Cast DPD           : 6.5s
Groupby            : 0.0s
Colonnes de travail: 8.3s


In [86]:
X.columns

Index(['mensualite_implicite', 'charge_taux_duree', 'pression_levier',
       'ecart_ltv_ocltv', 'couverture_mi', 'refi_flag', 'credit_segment',
       'primo_accedant', 'co_emprunteur', 'freq', 'niveau', 'ecart_au_plan',
       'anticipation'],
      dtype='object')

In [9]:
pipeline = FeaturePipeline(window_months=12,woe_config= {"iv_threshold": 0.01, "metric": "woe"})

In [12]:
x = pipeline.build(hist, orig)

Copy DataFrame     : 12.8s
Cast DPD           : 7.0s
Groupby            : 0.0s
Colonnes de travail: 10.6s


In [11]:
import src.PDcomponent.pipelines.pdFeaturePipeline as pdFeaturePipeline
importlib.reload(pdFeaturePipeline)

from src.PDcomponent.pipelines.pdFeaturePipeline import PDFeaturePipeline

In [12]:
pipeline = PDFeaturePipeline(window_months=12, woe_config={"iv_threshold": 0.01, "metric": "woe"})

In [13]:
x,y = pipeline.build(hist, orig)

Copy DataFrame     : 8.1s
Cast DPD           : 6.5s
Groupby            : 0.0s
Colonnes de travail: 11.6s


In [16]:
x.mensualite_implicite.unique()

array([ 0.0547619 , -0.88571429,  0.13809524, ...,  0.75823357,
        1.67835134, -0.26153023], shape=(7387,))

In [17]:
y.unique()

array([0, 1])